In [ ]:
import open3d as o3d
import numpy as np


In [ ]:


mesh = o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
mesh.compute_vertex_normals()
mesh.remove_duplicated_vertices()
edges = mesh.get_non_manifold_edges(allow_boundary_edges=False)

vertices=np.asarray(mesh.vertices)
raw_point_vertices= vertices[np.asarray(edges)]
points=np.unique(np.vstack(raw_point_vertices), axis=0)
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(points[250:550])
point_cloud.paint_uniform_color([1, 0, 0])  # Red
o3d.visualization.draw_geometries([mesh, point_cloud])
print(points[50:100])


[[ 1.01851854e+01  6.51851807e+01  6.63983762e-01]
 [ 1.04166660e+01  6.66666641e+01  6.40710711e-01]
 [ 1.04790411e+01  0.00000000e+00  2.52317250e-01]
 [ 1.06481476e+01  6.81481476e+01  6.15171015e-01]
 [ 1.08796301e+01  6.96296310e+01  5.87505162e-01]
 [ 1.11111116e+01  7.11111145e+01  5.57896078e-01]
 [ 1.13425922e+01  7.25925903e+01  5.26569843e-01]
 [ 1.15740738e+01  7.40740738e+01  4.93796110e-01]
 [ 1.18055553e+01  7.55555573e+01  4.59888816e-01]
 [ 1.19760475e+01  0.00000000e+00  1.95332393e-01]
 [ 1.20370378e+01  7.70370407e+01  4.25206363e-01]
 [ 1.22685184e+01  7.85185165e+01  3.90152186e-01]
 [ 1.25000000e+01  8.00000000e+01  3.55174184e-01]
 [ 1.27314816e+01  8.14814835e+01  3.20765346e-01]
 [ 1.29629622e+01  8.29629593e+01  2.87463635e-01]
 [ 1.31944447e+01  8.44444427e+01  2.55851030e-01]
 [ 1.34259262e+01  8.59259186e+01  2.26554155e-01]
 [ 1.34730539e+01  0.00000000e+00  1.36589453e-01]
 [ 1.36574078e+01  8.74074097e+01  2.00243086e-01]
 [ 1.38888884e+01  8.88888931e+

In [ ]:
#%%timeit -n 10 -r 5

mesh=o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
#mesh=o3d.io.read_triangle_mesh("plane_segments\Curvy.stl")
#mesh=o3d.io.read_triangle_mesh(r"plane_segments\Non-planar.stl")
mesh.compute_vertex_normals()
mesh.remove_duplicated_vertices()
edges_trial = mesh.get_non_manifold_edges(allow_boundary_edges=False)


edge_segments=np.asarray(mesh.vertices)[edges_trial]
edge_points=np.unique(edge_segments.reshape(-1,3), axis=0)

def order_perimeter(points, clockwise=True):
    #vectors = points - centroid
    #cross_prod = np.cross(vectors, np.array([0, 0, 1]))[:, 2]  # Sorting in 2D plane
    #sort_order = np.argsort(cross_prod)

    centroid=np.mean(points,axis=0)
    angles=np.arctan2(points[:,1]-centroid[1], points[:, 0]-centroid[0])
    sort_order = np.argsort(angles)
    
    if clockwise:
        sort_order = sort_order[::-1] 

    return points[sort_order]

def dot_product(points):

    forward_vecs=np.roll(points,shift=1, axis=0)-points
    backward_vecs=np.roll(points,shift=-1,axis=0)-points
    
    #einstine summation notation, more efficient than np.sum(f*b)
    dot_products=np.einsum("ij,ij->i",forward_vecs,backward_vecs)

    return dot_products

def split_perimeter(ordered_points, corner_indices):
    corner_indices = np.sort(corner_indices)  # Ensure indices are sorted
    segments=[]
    num_points=len(ordered_points)

    for i in range(len(corner_indices)):
        start_idx=corner_indices[i]
        end_idx=corner_indices[(i+1) % len(corner_indices)]
        if start_idx<end_idx:
            segment= ordered_points[start_idx: end_idx+1]
        else:
            segment=np.vstack([ordered_points[start_idx:],ordered_points[:end_idx+1]])
        segments.append(segment)
    return segments

def curvilinear_distance(segment):
    diffs=np.diff(segment,axis=0)
    distance=np.linalg.norm(diffs,axis=1)
    return(np.sum(distances))
    

ordered=order_perimeter(edge_points)
dps=dot_product(ordered)

k = 3  # Number of smallest values to retrieve
min_indices = np.argsort(np.abs(dps))[:4]
split=split_perimeter(ordered, min_indices)

#print("Indices with lowest absolute values:", min_indices)
# point_cloud = o3d.geometry.PointCloud()
# point_cloud.points = o3d.utility.Vector3dVector(split[2])
# point_cloud.paint_uniform_color([1, 0, 0])  # Red
# o3d.visualization.draw_geometries([mesh, point_cloud])






198 ms ± 4.25 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [117]:
x=np.array([[[1,2,3],[4,5,6]],[[7, 8, 9],[4, 5,6]]])

x=[1,2,3,4,5]
print(x[4:2])
print(np.roll(x,shift=-1,axis=2))

[]


AxisError: axis 2 is out of bounds for array of dimension 1